In [ ]:
# Usage requirements:
# 1. Run this in a local desktop Jupyter session, or in a remote Jupyter session with DISPLAY/GUI forwarding configured.
# 2. Install GUI dependencies with: pip install "cigvis[gui]". The machine must also have working OpenGL support.
# 3. Enable the Qt event loop before opening the VisPy canvas. Later plot3D uses run_app=False so the notebook cell is not blocked by vispy.app.run().
# 4. For pure SSH/headless environments, prefer plotlyplot, viserplot, or sliceviewer instead.
%gui qt
import vispy
vispy.use(app="pyside6")

In [ ]:
from pathlib import Path

import numpy as np
import cigvis
from cigvis import colormap

In [ ]:
# This mirrors examples/3Dvispy/04-overlay_rgt_fault.py, but opens a VisPy window from Jupyter.
# The volume order is line_first=True: volume.shape = (inline, crossline, time/depth).
root = Path("/Volumes/T7/DATA/cigvisdata/rgt3d")
ni, nx, nt = 128, 128, 128

seis = np.fromfile(root / "seis.dat", np.float32).reshape(ni, nx, nt)
rgt = np.fromfile(root / "rgt.dat", np.float32).reshape(ni, nx, nt)
fault = np.fromfile(root / "fault.dat", np.float32).reshape(ni, nx, nt)

In [ ]:
rgt_cmap = colormap.set_alpha("jet", 0.4)
# Mask the minimum value (0), where 0 means no fault.
fault_cmap = colormap.set_alpha_except_min("jet", alpha=1)

nodes = cigvis.create_slices(seis, pos=[[36], [28], [84]], cmap="gray")
nodes = cigvis.add_mask(nodes, rgt, cmap=rgt_cmap, interpolation="cubic")
# fault is discrete data, so use nearest interpolation.
nodes = cigvis.add_mask(nodes, fault, cmap=fault_cmap, interpolation="nearest")
# nodes += cigvis.create_colorbar_from_nodes(nodes, "RGT", select="mask", idx=0)

# In Jupyter, run_app=False is recommended because %gui qt keeps the Qt event loop responsive.
# In a normal .py script, you can omit run_app=False and use the default run_app=True.
canvas = cigvis.plot3D(
    nodes,
    view=cigvis.Plot3DView(size=(750, 600)),
    # gui=cigvis.Plot3DGui(enabled=False),
    gui=True,
    # run_app=False,
)
# canvas